# CNN Model Training

This notebook implements and trains a Convolutional Neural Network (CNN) for student face identification.

The model is developed from scratch using PyTorch and is trained on a curated dataset of student facial images. The objective is to learn discriminative facial features that enable accurate identification of individual students.

Training workflow:

* Data loading and transformation
* Train-validation split
* CNN architecture design
* Loss function selection
* Optimization and backpropagation
* Performance monitoring
* Model checkpoint saving

The trained model forms the core intelligence component of the VisionAttendAI project.


In [ ]:
import os
import sys
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import PIL
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import DataLoader, random_split
from torchinfo import summary
from torchvision import datasets, transforms
from tqdm.notebook import tqdm

torch.backends.cudnn.deterministic = True

In [5]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("matplotlib version:", matplotlib.__version__)
print("pandas version:", pd.__version__)
print("PIL version:", PIL.__version__)
print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)

Platform: win32
Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
---
matplotlib version: 3.10.0
pandas version: 2.2.3
PIL version: 11.1.0
torch version: 2.11.0+cpu
torchvision version: 0.26.0+cpu


In [8]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device.")

Using cpu device.


# Loading Data

In [9]:
data_dir = os.path.join("data_p1", "data_undersampled", "train")

print("Data directory:", data_dir)

Data directory: data_p1\data_undersampled\train


In [10]:
class ConvertToRGB:
    def __call__(self, img):
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

In [11]:
transform = transforms.Compose(
    [
        ConvertToRGB(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.6665, 0.6047, 0.5631], std=[0.2257, 0.2515, 0.2687])
    ]
)

In [12]:
dataset = datasets.ImageFolder(data_dir, transform)
dataset

Dataset ImageFolder
    Number of datapoints: 1200
    Root location: data_p1\data_undersampled\train
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.6665, 0.6047, 0.5631], std=[0.2257, 0.2515, 0.2687])
           )

In [13]:
classes = dataset.classes
classes

['abdulrahman', 'fatima', 'labaran', 'maryam', 'usman', 'zainab']

In [14]:
from training import class_counts

In [28]:
counts = class_counts(dataset)
counts

# Train-Validation Split

In [15]:
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])

length_train = len(train_dataset)
length_val = len(val_dataset)
length_dataset = len(dataset)
percent_train = np.round(100 * length_train / length_dataset, 2)
percent_val = np.round(100 * length_val / length_dataset, 2)

print(f"Train data is {percent_train}% of full data")
print(f"Validation data is {percent_val}% of full data")

Train data is 80.0% of full data
Validation data is 20.0% of full data


In [16]:
g = torch.Generator()
g.manual_seed(42)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

single_batch = next(iter(train_loader))[0]
print(f"Shape of one batch: {single_batch.shape}")

Shape of one batch: torch.Size([16, 3, 224, 224])


In [17]:
test_batch = next(iter(train_loader))[0]
batch_shape = test_batch.shape

print(f"Batch shape: {batch_shape}")

Batch shape: torch.Size([16, 3, 224, 224])


# Model Architecture

In [20]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model = torch.nn.Sequential()

conv1 = torch.nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3, 3), padding=1)
max_pool1 = torch.nn.MaxPool2d(kernel_size=(2, 2), stride=2)
model.append(conv1)
model.append(torch.nn.ReLU())
model.append(max_pool1)

conv2 = torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), padding=1)
max_pool2 = torch.nn.MaxPool2d(kernel_size=(2, 2), stride=2)
model.append(conv2)
model.append(torch.nn.ReLU())
model.append(max_pool2)

conv3 = torch.nn.Conv2d(32, 64, 3, padding=1)
max_pool3 = torch.nn.MaxPool2d(2)
model.append(conv3)
model.append(torch.nn.ReLU())
model.append(max_pool3)

model.append(torch.nn.Flatten())
model.append(torch.nn.Dropout())

linear1 = torch.nn.Linear(in_features=50176, out_features=500)
model.append(linear1)
model.append(torch.nn.ReLU())
model.append(torch.nn.Dropout())

output_layer = torch.nn.Linear(500, 6)
model.append(output_layer)

Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Dropout(p=0.5, inplace=False)
  (11): Linear(in_features=50176, out_features=500, bias=True)
  (12): ReLU()
  (13): Dropout(p=0.5, inplace=False)
  (14): Linear(in_features=500, out_features=6, bias=True)
)

In [21]:
height, width = 224, 224
summary(model, input_size=(batch_size, 3, height, width))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [16, 6]                   --
├─Conv2d: 1-1                            [16, 16, 224, 224]        448
├─ReLU: 1-2                              [16, 16, 224, 224]        --
├─MaxPool2d: 1-3                         [16, 16, 112, 112]        --
├─Conv2d: 1-4                            [16, 32, 112, 112]        4,640
├─ReLU: 1-5                              [16, 32, 112, 112]        --
├─MaxPool2d: 1-6                         [16, 32, 56, 56]          --
├─Conv2d: 1-7                            [16, 64, 56, 56]          18,496
├─ReLU: 1-8                              [16, 64, 56, 56]          --
├─MaxPool2d: 1-9                         [16, 64, 28, 28]          --
├─Flatten: 1-10                          [16, 50176]               --
├─Dropout: 1-11                          [16, 50176]               --
├─Linear: 1-12                           [16, 500]                 25,088,500

In [22]:
from training import predict, train

# Loss Function and Optimazation

In [23]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model.to(device)

rint(loss_fn)
print("----------------------")
print(optimizer)

Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Dropout(p=0.5, inplace=False)
  (11): Linear(in_features=50176, out_features=500, bias=True)
  (12): ReLU()
  (13): Dropout(p=0.5, inplace=False)
  (14): Linear(in_features=500, out_features=6, bias=True)
)

In [24]:
train?

Signature:
train(
    model,
    optimizer,
    loss_fn,
    train_loader,
    val_loader,
    epochs=20,
    device='cpu',
)
Docstring: <no docstring>
File:      c:\users\dimo\desktop\project\training.py
Type:      function

In [25]:
train(
    model,
    optimizer,
    loss_fn,
    train_loader,
    val_loader,
    epochs=3,
    device='cpu',
)

Training:   0%|          | 0/60 [00:00<?, ?it/s]

Scoring:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 1, Training Loss: 1.31, Validation Loss: 0.80, Validation accuracy = 0.75


Training:   0%|          | 0/60 [00:00<?, ?it/s]

Scoring:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 2, Training Loss: 0.54, Validation Loss: 0.59, Validation accuracy = 0.78


Training:   0%|          | 0/60 [00:00<?, ?it/s]

Scoring:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 3, Training Loss: 0.27, Validation Loss: 0.38, Validation accuracy = 0.88


In [28]:
train(
    model,
    optimizer,
    loss_fn,
    train_loader,
    val_loader,
    epochs=15,
    device='cpu',
)

Training:   0%|          | 0/60 [00:00<?, ?it/s]

Scoring:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 1, Training Loss: 0.12, Validation Loss: 0.45, Validation accuracy = 0.88


Training:   0%|          | 0/60 [00:00<?, ?it/s]

Scoring:   0%|          | 0/15 [00:00<?, ?it/s]

...

Scoring:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 14, Training Loss: 0.01, Validation Loss: 0.48, Validation accuracy = 0.90


Training:   0%|          | 0/60 [00:00<?, ?it/s]

Scoring:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 15, Training Loss: 0.01, Validation Loss: 0.40, Validation accuracy = 0.90


# Saving the model 

In [29]:
torch.save(model, "model/JAYset_1.0.pth")